# Chunk-to-Event Bridge with Cost-Aware Scaling

This notebook starts implementation of a **Chunk-to-Event Bridge** that converts chunk predictions into structured event records and adds cost-aware model routing.

## What this notebook implements
- Event schema (event type + arguments)
- Confidence and abstention for extracted events
- Cost profiles per model (including pending tiny models)
- Budget-aware model recommendation (quality mode vs latency mode)
- Email-to-event extraction pipeline

## Research Alignment (Web References)

- Luan et al. (NAACL 2019), Dynamic Span Graphs: https://aclanthology.org/N19-1308/
- Zhong & Chen (NAACL 2021), PURE: https://aclanthology.org/2021.naacl-main.5/
- Lu et al. (ACL 2022), UIE: https://aclanthology.org/2022.acl-long.395/
- Akbik et al. (COLING 2018), contextual sequence labeling: https://aclanthology.org/C18-1139/

Design direction: practical pipeline IE + structured event outputs + compute-aware routing.

In [1]:
from __future__ import annotations

import re
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", 160)

CHUNK_LABELS = [
    "O", "B-ADJP", "I-ADJP", "B-ADVP", "I-ADVP", "B-CONJP", "I-CONJP", "B-INTJ", "I-INTJ", "B-LST",
    "I-LST", "B-NP", "I-NP", "B-PP", "I-PP", "B-PRT", "I-PRT", "B-SBAR", "I-SBAR", "B-UCP", "I-UCP",
    "B-VP", "I-VP"
]

def find_project_root(start: Path) -> Path:
    cur = start.resolve()
    while cur != cur.parent:
        if (cur / "dataset").exists() and (cur / "ipynb").exists():
            return cur
        cur = cur.parent
    return start.resolve()

PROJECT_ROOT = find_project_root(Path.cwd())
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
print("Project root:", PROJECT_ROOT)
print("Outputs dir:", OUTPUTS_DIR)

Project root: /home/sg/dev/nlp
Outputs dir: /home/sg/dev/nlp/outputs


In [2]:
# Event schema and mapping rules
EVENT_KEYWORDS = {
    "meeting": ["meet", "meeting", "sync", "call", "discussion", "demo"],
    "deadline": ["deadline", "due", "submit", "delivery", "by"],
    "travel": ["travel", "flight", "hotel", "trip", "visit"],
    "incident": ["incident", "issue", "error", "outage", "failure"],
    "announcement": ["announce", "announcement", "release", "launched"],
}

ROLE_REQUIRED = {
    "meeting": ["trigger", "location_or_time"],
    "deadline": ["trigger", "time"],
    "travel": ["trigger", "location"],
    "incident": ["trigger"],
    "announcement": ["trigger", "subject"],
    "other": ["trigger"],
}

TIME_PATTERN = re.compile(
    r"\b(monday|tuesday|wednesday|thursday|friday|saturday|sunday|today|tomorrow|tonight|\d{1,2}[:.]\d{2}\s?(am|pm)?|\d{1,2}\s?(am|pm)|\d{1,2}[-/]\d{1,2}([-/]\d{2,4})?)\b",
    re.IGNORECASE,
)
LOCATION_PATTERN = re.compile(
    r"\b(in|at|from|to|near|inside|within|across|around|into|onto)\s+([A-Za-z0-9][A-Za-z0-9\-]*(?:\s+[A-Za-z0-9][A-Za-z0-9\-]*){0,6})",
    re.IGNORECASE,
)

def classify_event_type(trigger: Optional[str], sentence: str) -> str:
    text = f"{trigger or ''} {sentence or ''}".lower()
    for etype, keys in EVENT_KEYWORDS.items():
        if any(k in text for k in keys):
            return etype
    return "other"

In [3]:
# Chunk helpers and model loader
def bio_to_spans(tokens: List[str], chunk_labels: List[str]) -> List[Dict]:
    spans = []
    cur_type, cur_tokens, cur_start = None, [], None

    for i, (tok, tag) in enumerate(zip(tokens, chunk_labels)):
        if tag == "O":
            if cur_type is not None:
                spans.append({"type": cur_type, "text": " ".join(cur_tokens), "start": cur_start, "end": i - 1})
                cur_type, cur_tokens, cur_start = None, [], None
            continue

        prefix, ctype = tag.split("-", 1) if "-" in tag else ("B", tag)
        if prefix == "B" or ctype != cur_type:
            if cur_type is not None:
                spans.append({"type": cur_type, "text": " ".join(cur_tokens), "start": cur_start, "end": i - 1})
            cur_type, cur_tokens, cur_start = ctype, [tok], i
        else:
            cur_tokens.append(tok)

    if cur_type is not None:
        spans.append({"type": cur_type, "text": " ".join(cur_tokens), "start": cur_start, "end": len(tokens) - 1})

    return spans

def _normalize_label(tag: str) -> str:
    if not isinstance(tag, str) or not tag:
        return "O"
    t = tag.replace("LABEL_", "")
    if t.isdigit():
        idx = int(t)
        if 0 <= idx < len(CHUNK_LABELS):
            return CHUNK_LABELS[idx]
    return t

def _merge_subword_predictions(pred_items: List[Dict]) -> Tuple[List[str], List[str], List[float]]:
    merged_tokens, merged_labels, merged_scores = [], [], []

    for item in pred_items:
        token = str(item.get("word", ""))
        label = _normalize_label(item.get("entity") or item.get("entity_group") or "O")
        score = float(item.get("score", 0.5))

        is_subword = token.startswith("##")
        token_clean = token[2:] if is_subword else token
        if token_clean.startswith("\u0120") or token_clean.startswith("\u2581"):
            token_clean = token_clean[1:]

        if is_subword and merged_tokens:
            merged_tokens[-1] += token_clean
            merged_scores[-1] = min(merged_scores[-1], score)
            continue

        merged_tokens.append(token_clean)
        merged_labels.append(label)
        merged_scores.append(score)

    n = min(len(merged_tokens), len(merged_labels), len(merged_scores))
    return merged_tokens[:n], merged_labels[:n], merged_scores[:n]

def pick_latest_checkpoint(base_dir: Path) -> Optional[Path]:
    if not base_dir.exists():
        return None
    ckpts = [
        p for p in base_dir.iterdir()
        if p.is_dir() and p.name.startswith("checkpoint-") and (p / "config.json").exists()
    ]
    if ckpts:
        return sorted(ckpts, key=lambda p: int(p.name.split("-")[-1]))[-1]
    if (base_dir / "config.json").exists():
        return base_dir
    return None

def load_chunk_pipeline(model_key: str):
    from transformers import pipeline

    model_roots = {
        "distilbert": OUTPUTS_DIR / "distilbert-conll2000",
        "bert-base-uncased": OUTPUTS_DIR / "scale-study-bert-base-uncased",
        "roberta-base": OUTPUTS_DIR / "scale-study-roberta-base",
        "google/bert_uncased_L-2_H-128_A-2": OUTPUTS_DIR / "scale-study-bert_uncased_L-2_H-128_A-2",
        "prajjwal1/bert-tiny": OUTPUTS_DIR / "scale-study-bert-tiny",
        "huawei-noah/TinyBERT_General_4L_312D": OUTPUTS_DIR / "scale-study-TinyBERT_General_4L_312D",
    }

    base = model_roots.get(model_key)
    if base is None:
        raise KeyError(f"Unknown model key: {model_key}")

    ckpt = pick_latest_checkpoint(base)
    if ckpt is None:
        raise FileNotFoundError(f"Checkpoint not found for {model_key}: {base}")

    print("Loading checkpoint:", ckpt)
    return pipeline(
        "token-classification",
        model=str(ckpt),
        tokenizer=str(ckpt),
        aggregation_strategy="none",
    )

In [4]:
# Event extraction + confidence + abstention
def sentence_to_ie_record(tokens: List[str], chunk_labels: List[str]) -> Dict:
    spans = bio_to_spans(tokens, chunk_labels)
    nps = [s for s in spans if s["type"] == "NP"]
    vps = [s for s in spans if s["type"] == "VP"]

    subject = nps[0]["text"] if nps else None
    action = vps[0]["text"] if vps else None

    obj = None
    if vps:
        vp_end = vps[0]["end"]
        for np_span in nps:
            if np_span["start"] > vp_end:
                obj = np_span["text"]
                break

    return {
        "subject": subject,
        "action": action,
        "object": obj,
        "num_chunks": len(spans),
        "spans": spans,
    }

def extract_location_phrases(sentence: str) -> List[str]:
    found = []
    for m in LOCATION_PATTERN.finditer(sentence or ""):
        phrase = f"{m.group(1)} {m.group(2)}".strip()
        if TIME_PATTERN.search(phrase):
            continue
        found.append(phrase)
    return list(dict.fromkeys([x.lower() for x in found]))

def extract_time_phrases(sentence: str) -> List[str]:
    found = [m.group(0).strip() for m in TIME_PATTERN.finditer(sentence or "")]
    return list(dict.fromkeys([x.lower() for x in found]))

def role_completeness_score(event_type: str, subject: Optional[str], trigger: Optional[str], obj: Optional[str], locations: List[str], times: List[str]) -> float:
    has_location_or_time = bool(locations or times)
    provided = {
        "trigger": bool(trigger),
        "subject": bool(subject),
        "object": bool(obj),
        "location": bool(locations),
        "time": bool(times),
        "location_or_time": has_location_or_time,
    }
    required = ROLE_REQUIRED.get(event_type, ["trigger"])
    ok = sum(1 for r in required if provided.get(r, False))
    return ok / max(len(required), 1)

def extract_event_record(sentence: str, pipeline_obj, confidence_threshold: float = 0.55) -> Dict:
    pred = pipeline_obj(sentence)
    tokens, labels, scores = _merge_subword_predictions(pred)

    if not tokens:
        return {
            "sentence": sentence,
            "event_type": "other",
            "trigger": None,
            "subject": None,
            "object": None,
            "location": None,
            "time": None,
            "confidence": 0.0,
            "abstained": True,
            "abstain_reason": "no_tokens",
            "num_chunks": 0,
            "tokens_processed": 0,
        }

    ie = sentence_to_ie_record(tokens, labels)
    trigger = ie["action"]
    event_type = classify_event_type(trigger, sentence)

    locations = extract_location_phrases(sentence)
    times = extract_time_phrases(sentence)

    mean_token_conf = float(np.mean(scores)) if scores else 0.5
    role_score = role_completeness_score(event_type, ie["subject"], trigger, ie["object"], locations, times)
    lexical_bonus = 1.0 if event_type != "other" else 0.7
    confidence = float(np.clip(0.55 * mean_token_conf + 0.35 * role_score + 0.10 * lexical_bonus, 0.0, 1.0))

    abstained = confidence < confidence_threshold
    abstain_reason = "low_confidence" if abstained else None

    return {
        "sentence": sentence,
        "event_type": event_type,
        "trigger": trigger,
        "subject": ie["subject"],
        "object": ie["object"],
        "location": "; ".join(locations) if locations else None,
        "time": "; ".join(times) if times else None,
        "confidence": confidence,
        "abstained": abstained,
        "abstain_reason": abstain_reason,
        "num_chunks": ie["num_chunks"],
        "tokens_processed": len(tokens),
    }

In [5]:
# Cost-aware scaling profile (seeded from existing results + pending tiny models)
model_cost_profile = pd.DataFrame([
    {"model": "distilbert", "chunk_f1": 0.9566, "train_seconds": 120.0704, "params_millions": 66.3806, "status": "measured"},
    {"model": "bert-base-uncased", "chunk_f1": 0.9602, "train_seconds": 222.7203, "params_millions": 108.9093, "status": "measured"},
    {"model": "roberta-base", "chunk_f1": 0.9665, "train_seconds": 224.6860, "params_millions": 124.0727, "status": "measured"},
    {"model": "google/bert_uncased_L-2_H-128_A-2", "chunk_f1": np.nan, "train_seconds": np.nan, "params_millions": np.nan, "status": "pending_evaluation"},
    {"model": "prajjwal1/bert-tiny", "chunk_f1": np.nan, "train_seconds": np.nan, "params_millions": np.nan, "status": "pending_evaluation"},
    {"model": "huawei-noah/TinyBERT_General_4L_312D", "chunk_f1": np.nan, "train_seconds": np.nan, "params_millions": np.nan, "status": "pending_evaluation"},
])

base = model_cost_profile[model_cost_profile["model"] == "distilbert"].iloc[0]
model_cost_profile["f1_gain_vs_base"] = model_cost_profile["chunk_f1"] - float(base["chunk_f1"])
model_cost_profile["train_time_ratio_vs_base"] = model_cost_profile["train_seconds"] / float(base["train_seconds"])

display(model_cost_profile.round(4))

,model,chunk_f1,train_seconds,params_millions,status,f1_gain_vs_base,train_time_ratio_vs_base
0,distilbert,0.9566,120.0704,66.3806,measured,0.0000,1.0000
1,bert-base-uncased,0.9602,222.7203,108.9093,measured,0.0036,1.8549
2,roberta-base,0.9665,224.6860,124.0727,measured,0.0099,1.8713
3,google/bert_uncased_L-2_H-128_A-2,NaN,NaN,NaN,pending_evaluation,NaN,NaN
4,prajjwal1/bert-tiny,NaN,NaN,NaN,pending_evaluation,NaN,NaN
5,huawei-noah/TinyBERT_General_4L_312D,NaN,NaN,NaN,pending_evaluation,NaN,NaN


In [6]:
# Model recommendation policy
def recommend_model(max_time_multiplier: float = 2.0, min_f1_gain: float = 0.005, objective: str = "quality") -> pd.DataFrame:
    df = model_cost_profile.copy()
    measured = df[df["status"] == "measured"].copy()

    feasible = measured[
        (measured["train_time_ratio_vs_base"] <= max_time_multiplier) &
        (measured["f1_gain_vs_base"] >= min_f1_gain)
    ].copy()

    if objective == "latency":
        feasible = feasible.sort_values(["train_seconds", "chunk_f1"], ascending=[True, False])
    else:
        feasible = feasible.sort_values(["chunk_f1", "train_seconds"], ascending=[False, True])

    pending = df[df["status"] != "measured"].copy()
    pending["note"] = "train/evaluate pending"

    return feasible[["model", "chunk_f1", "f1_gain_vs_base", "train_time_ratio_vs_base", "status"]], pending[["model", "status", "note"]]

quality_rec, pending_models = recommend_model(objective="quality")
latency_rec, _ = recommend_model(objective="latency")

print("Quality-first recommendation candidates:")
display(quality_rec.round(4))
print("\nLatency-first recommendation candidates:")
display(latency_rec.round(4))
print("\nPending tiny models:")
display(pending_models)

Quality-first recommendation candidates:


,model,chunk_f1,f1_gain_vs_base,train_time_ratio_vs_base,status
2,roberta-base,0.9665,0.0099,1.8713,measured



Latency-first recommendation candidates:


,model,chunk_f1,f1_gain_vs_base,train_time_ratio_vs_base,status
2,roberta-base,0.9665,0.0099,1.8713,measured



Pending tiny models:


,model,status,note
3,google/bert_uncased_L-2_H-128_A-2,pending_evaluation,train/evaluate pending
4,prajjwal1/bert-tiny,pending_evaluation,train/evaluate pending
5,huawei-noah/TinyBERT_General_4L_312D,pending_evaluation,train/evaluate pending


In [7]:
# Email pipeline with cost accounting
def split_email_into_sentences(email_text: str) -> List[str]:
    lines = [ln.strip() for ln in (email_text or "").replace("\r", "").split("\n") if ln.strip()]
    lines = [ln for ln in lines if not re.match(r"^(subject|from|to|cc|bcc):", ln, flags=re.IGNORECASE)]
    sents = []
    for ln in lines:
        sents.extend([p.strip() for p in re.split(r"(?<=[.!?])\s+", ln) if p.strip()])
    return sents

def estimate_event_cost(tokens_processed: int, model_name: str) -> float:
    row = model_cost_profile[model_cost_profile["model"] == model_name]
    if row.empty or pd.isna(row.iloc[0]["params_millions"]):
        return np.nan
    params_scale = float(row.iloc[0]["params_millions"]) / float(base["params_millions"])
    # Relative proxy cost units, not wall-clock ms.
    return float(tokens_processed * params_scale)

def run_email_event_pipeline(email_text: str, model_name: str = "distilbert", confidence_threshold: float = 0.55) -> pd.DataFrame:
    pipe = load_chunk_pipeline(model_name)
    rows = []
    for sent in split_email_into_sentences(email_text):
        rec = extract_event_record(sent, pipe, confidence_threshold=confidence_threshold)
        rec["model"] = model_name
        rec["cost_estimate"] = estimate_event_cost(rec["tokens_processed"], model_name)
        rows.append(rec)

    df = pd.DataFrame(rows)
    if not df.empty:
        df["accepted"] = ~df["abstained"]
    return df

email_text = """
Subject: Release readiness and office move
Hi team,
Let's meet in Conference Room 3 at 4:00 PM tomorrow to finalize launch blockers.
Please submit your checklist by Monday evening.
Operations will move hardware to the new office next week.
Thanks.
""".strip()

# Run with distilbert by default.
# Change model_name to measured alternatives after checkpoints are available.
try:
    event_df = run_email_event_pipeline(email_text, model_name="distilbert", confidence_threshold=0.58)
    display(event_df[["sentence", "event_type", "trigger", "subject", "object", "location", "time", "confidence", "abstained", "cost_estimate"]])

    if not event_df.empty:
        accepted = event_df[~event_df["abstained"]]
        total_cost = float(event_df["cost_estimate"].sum(skipna=True))
        accepted_count = int(len(accepted))
        cost_per_accepted = total_cost / max(accepted_count, 1)

        print(f"Accepted events: {accepted_count}/{len(event_df)}")
        print(f"Total cost estimate: {total_cost:.2f}")
        print(f"Cost per accepted event: {cost_per_accepted:.2f}")
except Exception as e:
    print("Pipeline run skipped or failed.")
    print("Reason:", e)

/home/sg/dev/nlp/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading checkpoint: /home/sg/dev/nlp/outputs/distilbert-conll2000/checkpoint-1677


Loading weights: 100%|██████████| 102/102 [00:00<00:00, 5948.71it/s]


,sentence,event_type,trigger,subject,object,location,time,confidence,abstained,cost_estimate
0,"Hi team,",other,NaN,hi team,NaN,NaN,NaN,0.612629,False,2.0
1,Let's meet in Conference Room 3 at 4:00 PM tomorrow to finalize launch blockers.,meeting,',let,conference room 3,in conference room 3 at 4; to finalize launch blockers,4:00 pm; tomorrow,0.981277,False,18.0
2,Please submit your checklist by Monday evening.,deadline,submit,your checklist,your checklist,NaN,monday,0.948421,False,7.0
3,Operations will move hardware to the new office next week.,other,will move,operations,hardware,to the new office next week,NaN,0.968835,False,10.0
4,Thanks.,other,NaN,thanks,NaN,NaN,NaN,0.582001,False,1.0


Accepted events: 5/5
Total cost estimate: 38.00
Cost per accepted event: 7.60


## Next Build Steps

1. Replace proxy `cost_estimate` with measured latency/token from actual inference profiling.
2. Evaluate tiny models after training and update `model_cost_profile`.
3. Add event-level precision/recall/F1 against a labeled event dataset.
4. Export accepted events as JSONL for downstream workflow integration.